In [17]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("Dataset.csv")

In [18]:
# ------------------------------------------
# 1. BASIC DATA CLEANING
# ------------------------------------------

# Remove duplicates
df.drop_duplicates(inplace=True)

# Standardize column names
df.columns = df.columns.str.lower().str.replace(" ", "_")

# Check missing values
print(df.isnull().sum())

customer_id                0
age                        0
gender                     0
item_purchased             0
category                   0
purchase_amount_(usd)      0
location                   0
size                       0
color                      0
season                     0
review_rating             37
subscription_status        0
shipping_type              0
discount_applied           0
promo_code_used            0
previous_purchases         0
payment_method             0
frequency_of_purchases     0
dtype: int64


In [19]:
# ------------------------------------------
# 2. ENCODE YES/NO COLUMNS
# ------------------------------------------

binary_cols = [
    "subscription_status",
    "discount_applied",
    "promo_code_used"
]

for col in binary_cols:
    df[col] = df[col].map({"Yes": 1, "No": 0})

In [20]:
# ------------------------------------------
# 3. CUSTOMER VALUE FEATURES
# ------------------------------------------

# Total customer spending proxy
# Higher purchase amount + higher previous purchases = higher value
df["customer_value_score"] = (
    df["purchase_amount_(usd)"] * df["previous_purchases"]
)

# Average spend per previous purchase
df["avg_spend_per_purchase"] = (
    df["purchase_amount_(usd)"] /
    (df["previous_purchases"] + 1)
)

# High spender flag
df["high_spender"] = np.where(
    df["purchase_amount_(usd)"] >
    df["purchase_amount_(usd)"].median(),
    1,
    0
)

In [21]:
# ------------------------------------------
# 4. CUSTOMER LOYALTY FEATURES
# ------------------------------------------

# Loyalty score
df["loyalty_score"] = (
    df["previous_purchases"] +
    (df["subscription_status"] * 10)
)

# Repeat customer
df["repeat_customer"] = np.where(
    df["previous_purchases"] >= 10,
    1,
    0
)

In [22]:
# ------------------------------------------
# 5. PROMOTION DEPENDENCY FEATURES
# ------------------------------------------

# Customer depends on discounts/promos
df["promotion_dependency"] = (
    df["discount_applied"] +
    df["promo_code_used"]
)

# Strong promo reliance
df["high_promo_user"] = np.where(
    df["promotion_dependency"] == 2,
    1,
    0
)

In [23]:
# ------------------------------------------
# 6. CUSTOMER SATISFACTION FEATURES
# ------------------------------------------

# Rating buckets
df["rating_segment"] = pd.cut(
    df["review_rating"],
    bins=[0, 2.5, 3.5, 5],
    labels=["Low", "Medium", "High"]
)

# Dissatisfied customer
df["low_rating_flag"] = np.where(
    df["review_rating"] < 3,
    1,
    0
)

In [24]:
# ------------------------------------------
# 7. PURCHASE FREQUENCY FEATURES
# ------------------------------------------

frequency_map = {
    "Weekly": 52,
    "Fortnightly": 26,
    "Monthly": 12,
    "Quarterly": 4,
    "Annually": 1,
    "Bi-Weekly": 26,
    "Every 3 Months": 4
}

df["purchase_frequency_score"] = (
    df["frequency_of_purchases"]
    .map(frequency_map)
    .fillna(0)
)

In [25]:
# ------------------------------------------
# 8. AGE-BASED FEATURES
# ------------------------------------------

# Age groups
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 18, 25, 35, 50, 100],
    labels=["Teen", "Young Adult", "Adult", "Middle Age", "Senior"]
)

In [26]:
# ------------------------------------------
# 9. SEASONAL SHOPPING FEATURES
# ------------------------------------------

# Seasonal category demand
df["season_category"] = (
    df["season"] + "_" + df["category"]
)

In [27]:

# ------------------------------------------
# 10. SHIPPING BEHAVIOR FEATURES
# ------------------------------------------

# Fast shipping preference
fast_shipping = [
    "Express",
    "Next Day Air",
    "2-Day Shipping"
]

df["fast_shipping_user"] = np.where(
    df["shipping_type"].isin(fast_shipping),
    1,
    0
)

# ------------------------------------------
# 11. PAYMENT BEHAVIOR FEATURES
# ------------------------------------------

# Digital payment usage
digital_payments = [
    "PayPal",
    "Venmo",
    "Credit Card"
]

df["digital_payment_user"] = np.where(
    df["payment_method"].isin(digital_payments),
    1,
    0
)

# ------------------------------------------
# 12. ONE-HOT ENCODING
# ------------------------------------------

categorical_cols = [
    "gender",
    "category",
    "season",
    "shipping_type",
    "payment_method",
    "age_group"
]

df = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True
)

# ------------------------------------------
# 13. FINAL DATASET INFO
# ------------------------------------------

print(df.head())
print("\nFinal Shape:", df.shape)

# Save engineered dataset
df.to_csv("engineered_customer_dataset.csv", index=False)

print("\nFeature engineering completed successfully!")

   customer_id  age item_purchased  purchase_amount_(usd)       location size  \
0            1   55         Blouse                     53       Kentucky    L   
1            2   19        Sweater                     64          Maine    L   
2            3   50          Jeans                     73  Massachusetts    S   
3            4   21        Sandals                     90   Rhode Island    M   
4            5   45         Blouse                     49         Oregon    M   

       color  review_rating  subscription_status  discount_applied  ...  \
0       Gray            3.1                    1                 1  ...   
1     Maroon            3.1                    1                 1  ...   
2     Maroon            3.1                    1                 1  ...   
3     Maroon            3.5                    1                 1  ...   
4  Turquoise            2.7                    1                 1  ...   

   shipping_type_Store Pickup  payment_method_Cash payment_met